# Australian Regional Temperature Amplification

Computes the ratio of southeastern Australian warming to global mean warming from the CMIP6 multi-model ensemble. This amplification factor tightens the link between entity global warming contributions (from FaIR) and Australian regional fire weather risk.

**Method**: stream CMIP6 `historical` surface air temperature (`tas`) from pangeo for 7 models. Compute area-weighted annual means for (a) global and (b) SE Australia. Fit linear trends 1900–2014. Amplification = SE_AU trend / global trend.

**SE Australia region**: lat −44° to −28°S, lon 138° to 154°E — covering the main fire-affected states (NSW, VIC, SA) consistent with the WWA Black Summer study region.

**Output**: `data/processed/entity_warming_contribution.parquet` updated with `warming_au_*` columns.

> **Status note (2026-06-13 methodology revision).** This notebook's amplification factor is
> an **annual-mean `tas`** ratio (not fire-season tasmax). It is a *sensitivity* input only.
> The primary regional response used in attribution is the ERA5-observed **fire-season**
> amplification (0.726, notebook 05). After the apportionment fix, the final liability uses each
> entity's **global** warming share directly; the `warming_au_*` columns written here are
> **diagnostic only** and no longer multiply into the liability. The amplification factor enters
> the Probability Ratio as the additive shift coefficient β in the GEV shift-fit (notebook 04),
> where β=0.726 is primary and this CMIP6 value (0.935) is a sensitivity.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xarray as xr
import intake
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

PROC = Path('../../data/processed')
FIGS = Path('../../outputs/figures')

# SE Australia bounding box (matches WWA Black Summer study region)
LAT_S, LAT_N = -44, -28
LON_W, LON_E = 138, 154

# Baseline and trend periods
BASELINE = slice('1850', '1900')
TREND    = slice('1901', '2014')

# Use 3 models: both Australian models + one well-validated global model
# Opened one at a time to avoid slow simultaneous zarr opens
MODELS = ['ACCESS-CM2', 'ACCESS-ESM1-5', 'MPI-ESM1-2-HR']

## 1. Load CMIP6 catalog and fetch model data

We stream spatial means only — no full-grid downloads. Each model produces two time series (global, SE Australia).

In [ ]:
col = intake.open_esm_datastore('https://storage.googleapis.com/cmip6/pangeo-cmip6.json')

cat = col.search(
    variable_id='tas',
    experiment_id='historical',
    table_id='Amon',
    source_id=MODELS,
    member_id='r1i1p1f1',
)
print(f'Found {len(cat.df)} dataset entries')
print(cat.df[['source_id', 'member_id', 'grid_label']].to_string(index=False))

In [ ]:
def area_weighted_mean(ds, var='tas', lat_s=None, lat_n=None, lon_w=None, lon_e=None):
    """Compute cosine-latitude weighted spatial mean, optionally over a bounding box."""
    da = ds[var]

    # Drop any extra dimensions (e.g. member_id, dcpp_init_year) that survive the catalog filter
    extra_dims = [d for d in da.dims if d not in ('time', 'lat', 'lon')]
    if extra_dims:
        da = da.isel({d: 0 for d in extra_dims})

    # Normalise longitudes to 0–360 for consistent slicing
    if lon_w is not None:
        if da.lon.values.min() < 0:
            da = da.assign_coords(lon=(da.lon % 360)).sortby('lon')
        lon_w_norm = lon_w % 360
        lon_e_norm = lon_e % 360
        da = da.sel(lat=slice(lat_s, lat_n), lon=slice(lon_w_norm, lon_e_norm))
    elif lat_s is not None:
        da = da.sel(lat=slice(lat_s, lat_n))

    weights = np.cos(np.deg2rad(da.lat)).broadcast_like(da)
    return da.weighted(weights).mean(dim=['lat', 'lon']).squeeze()


def annual_anomaly(ts, baseline=BASELINE):
    """Resample monthly to annual, compute anomaly relative to baseline."""
    annual = ts.resample(time='YE').mean()
    base_mean = annual.sel(time=baseline).mean('time')
    return annual - base_mean


def linear_trend(ts, period=TREND):
    """Return OLS trend (°C/year) over the specified period as a Python float."""
    sub = ts.sel(time=period).dropna('time')
    # Ensure 1-D after any residual squeeze
    vals = np.asarray(sub.values).flatten()
    years = sub.time.dt.year.values.astype(float)
    slope, _, _, _, _ = stats.linregress(years, vals)
    return float(slope)

In [ ]:
CATALOG_LOCAL = Path('../../data/processed/pangeo-cmip6.json').resolve()

print(f'Loading catalog from local cache: {CATALOG_LOCAL}')
col = intake.open_esm_datastore(str(CATALOG_LOCAL))
print('Catalog loaded.')

cat = col.search(
    variable_id='tas',
    experiment_id='historical',
    table_id='Amon',
    source_id=MODELS,
    member_id='r1i1p1f1',
)
print(f'Found {len(cat.df)} entries:')
print(cat.df[['source_id', 'member_id', 'grid_label', 'zstore']].to_string(index=False))

In [ ]:
results = []

for _, row in cat.df.iterrows():
    model = row['source_id']
    zstore = row['zstore']
    print(f'Processing {model}...', end=' ', flush=True)
    try:
        # Open one zarr store at a time — much faster than to_dataset_dict
        ds = xr.open_zarr(zstore, consolidated=True)

        ts_global = area_weighted_mean(ds)
        anom_global = annual_anomaly(ts_global)
        trend_global = linear_trend(anom_global)

        ts_au = area_weighted_mean(ds, lat_s=LAT_S, lat_n=LAT_N, lon_w=LON_W, lon_e=LON_E)
        anom_au = annual_anomaly(ts_au)
        trend_au = linear_trend(anom_au)

        amplification = float(trend_au / trend_global)

        results.append({
            'model': model,
            'trend_global_degC_per_yr': trend_global,
            'trend_au_degC_per_yr': trend_au,
            'amplification': amplification,
            'anom_global': anom_global,
            'anom_au': anom_au,
        })
        print(f'amplification = {amplification:.3f}')
    except Exception as e:
        print(f'FAILED: {e}')

print(f'\nSuccessfully processed {len(results)}/{len(cat.df)} models')

## 2. Ensemble amplification factor

In [ ]:
amps = pd.DataFrame([{
    'model': r['model'],
    'trend_global': r['trend_global_degC_per_yr'] * 100,  # °C/century
    'trend_au': r['trend_au_degC_per_yr'] * 100,
    'amplification': r['amplification'],
} for r in results])

amp_p50 = float(np.median(amps['amplification']))
amp_p05 = float(np.percentile(amps['amplification'], 5))
amp_p95 = float(np.percentile(amps['amplification'], 95))

print('SE Australia amplification factor (trend ratio vs global):')  
print(f'  Median (p50): {amp_p50:.3f}')
print(f'  5th–95th:     [{amp_p05:.3f}, {amp_p95:.3f}]')
print()
print('Per-model breakdown:')
print(amps[['model', 'trend_global', 'trend_au', 'amplification']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: temperature anomaly time series for each model
ax = axes[0]
trend_period_years = range(1901, 2015)
for r in results:
    anom_au = r['anom_au'].sel(time=TREND)
    years = anom_au.time.dt.year.values
    ax.plot(years, anom_au.values, alpha=0.5, linewidth=1, label=r['model'])
ax.axhline(0, color='k', linewidth=0.5, linestyle='--')
ax.set_title('SE Australia temperature anomaly — CMIP6 historical', fontsize=11)
ax.set_xlabel('Year')
ax.set_ylabel('°C anomaly (vs 1850–1900)')
ax.legend(fontsize=7, ncol=2)

# Right: amplification factor per model with ensemble range
ax2 = axes[1]
colors = ['#FF5722' if 'ACCESS' in m else '#2196F3' for m in amps['model']]
bars = ax2.barh(amps['model'], amps['amplification'], color=colors, alpha=0.8)
ax2.axvline(amp_p50, color='k', linewidth=1.5, linestyle='--', label=f'Median = {amp_p50:.2f}')
ax2.axvspan(amp_p05, amp_p95, alpha=0.1, color='k', label=f'5–95th [{amp_p05:.2f}, {amp_p95:.2f}]')
ax2.axvline(1.0, color='grey', linewidth=0.8, linestyle=':')
ax2.set_xlabel('Amplification factor (SE Australia / global trend)')
ax2.set_title('SE Australia warming amplification by model', fontsize=11)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig(FIGS / 'au_regional_amplification.png', bbox_inches='tight')
plt.show()

## 3. Apply to entity warming contributions

Add Australian-specific warming columns to `entity_warming_contribution.parquet`. Entity shares within Carbon Majors are unchanged (CO2 is globally well-mixed, so proportional contributions are the same regionally). The amplification scales the *magnitude* of warming attributable to each entity in an Australian context.

In [ ]:
ew = pd.read_parquet(PROC / 'entity_warming_contribution.parquet')

ew['warming_au_p50_degC'] = ew['warming_p50_degC'] * amp_p50
ew['warming_au_p05_degC'] = ew['warming_p05_degC'] * amp_p05
ew['warming_au_p95_degC'] = ew['warming_p95_degC'] * amp_p95

print(f'Amplification factor applied: {amp_p50:.3f} [p05={amp_p05:.3f}, p95={amp_p95:.3f}]')
print(f'\nTop 10 entities — SE Australia attributed warming (m°C, p50):')
top10 = ew.nlargest(10, 'warming_au_p50_degC')[[
    'parent_entity', 'parent_type',
    'warming_p50_degC', 'warming_au_p50_degC'
]].copy()
top10['global_mdegC']    = top10['warming_p50_degC'] * 1000
top10['au_mdegC'] = top10['warming_au_p50_degC'] * 1000
print(top10[['parent_entity', 'parent_type', 'global_mdegC', 'au_mdegC']].to_string(index=False))

In [ ]:
# How does amplification change the total Carbon Majors attributed warming for Australia?
total_global = ew['warming_p50_degC'].sum()
total_au     = ew['warming_au_p50_degC'].sum()

print(f'Carbon Majors total attributed warming:')
print(f'  Global mean:       {total_global*1000:.1f} m°C ({total_global:.4f} °C)')
print(f'  SE Australia (×{amp_p50:.2f}): {total_au*1000:.1f} m°C ({total_au:.4f} °C)')
print()
print(f'Note: entity SHARES are unchanged — amplification scales magnitude only.')
print(f'Liability fractions are identical; Australian warming framing is more contextually relevant.')

## 4. Updated Black Summer liability with Australian framing

The liability *fractions* are unchanged, but we can now express entity warming in Australian terms and add the amplification uncertainty to the uncertainty budget.

In [ ]:
lb = pd.read_parquet(PROC / 'black_summer_liability.parquet')

# Merge in the Australian warming columns
lb = lb.merge(
    ew[['parent_entity', 'warming_au_p50_degC', 'warming_au_p05_degC', 'warming_au_p95_degC']],
    on='parent_entity', how='left'
)

# Recompute central liability uncertainty including both FaIR ensemble AND amplification uncertainty
AUD_TO_USD = 0.69
d_central  = 10.0 * AUD_TO_USD
far_central = 1 - 1/9

total_au_p50 = ew['warming_au_p50_degC'].sum()
total_au_p05 = ew['warming_au_p05_degC'].sum()
total_au_p95 = ew['warming_au_p95_degC'].sum()

lb['au_warming_share']     = lb['warming_au_p50_degC'] / total_au_p50
lb['liability_au_USD_M']   = lb['au_warming_share'] * far_central * d_central * 1000
lb['liability_au_p05_USD_M'] = (lb['warming_au_p05_degC'] / total_au_p05) * far_central * d_central * 1000
lb['liability_au_p95_USD_M'] = (lb['warming_au_p95_degC'] / total_au_p95) * far_central * d_central * 1000

lb = lb.sort_values('liability_au_USD_M', ascending=False).reset_index(drop=True)

print('Top 10 — central scenario (USD M), original vs Australian-framed:')
cmp = lb.head(10)[['parent_entity', 'liability_central_USD_M', 'liability_au_USD_M',
                    'liability_au_p05_USD_M', 'liability_au_p95_USD_M']].copy()
cmp.columns = ['Entity', 'Original $M', 'AU-framed $M', 'p05 $M', 'p95 $M']
for c in ['Original $M', 'AU-framed $M', 'p05 $M', 'p95 $M']:
    cmp[c] = cmp[c].map('{:,.1f}'.format)
print(cmp.to_string(index=False))

In [ ]:
# Save updated files
ew.to_parquet(PROC / 'entity_warming_contribution.parquet', index=False)
lb.to_parquet(PROC / 'black_summer_liability.parquet', index=False)

# Save amplification table
amps[['model', 'trend_global', 'trend_au', 'amplification']].to_csv(
    PROC / 'au_amplification_factor.csv', index=False
)

print('Saved:')
print(f'  entity_warming_contribution.parquet  — added warming_au_p05/p50/p95_degC columns')
print(f'  black_summer_liability.parquet        — added au_warming_share, liability_au_* columns')
print(f'  au_amplification_factor.csv           — per-model amplification breakdown')
print()
print(f'Amplification factor: {amp_p50:.3f} [5–95th: {amp_p05:.3f}–{amp_p95:.3f}]')

## Key findings

- **Amplification factor**: 0.935 (range: 0.841–1.030 across 2 models) — SE Australia warmed at ~0.94× the global mean rate, 1901–2014
- **Entity shares**: unchanged — proportionality holds for well-mixed GHGs; all entities scaled by the same factor
- **Australian framing (top entities, m°C)**:

  | Entity | Global (m°C) | SE Australia ×0.93 (m°C) |
  |--------|-------------|--------------------------|
  | Saudi Aramco | 44.7 | 41.8 |
  | ExxonMobil | 37.6 | 35.2 |
  | Gazprom | 32.9 | 30.8 |

- **Biggest inter-model spread**: ACCESS-ESM1-5 (0.841) vs ACCESS-CM2 (1.030)
- **Key caveat**: CMIP6 models underestimate observed Australian warming. BoM data implies amplification ~1.35 (Australia warmed ~1.47°C vs ~1.09°C globally since 1910). Our CMIP6-derived estimates are **conservative lower bounds** — consistent with the WWA Black Summer study's own caveat.

→ See `wiki/findings/2026-05-23-australia-regional-amplification.md`